# Week 99, Day 4: Mock loop

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w099d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import copy
grid = [[0, 0], [0, 0]]
shallow, deep = grid.copy(), copy.deepcopy(grid)
grid[0][0] = 9
print(shallow[0][0], deep[0][0])

*Interactive figure on the website: Two and a half hours for the whole loop..*

### Exercise: Coding 1: A spiral walk

Write `spiral(matrix)`: the values of a rectangular matrix in clockwise spiral order from the top left.

In [ ]:
def spiral(matrix):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_spiral():\n    \"round and in\"\n    assert spiral([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]\n    assert spiral([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]\n    assert spiral([]) == [] and spiral([[1], [2], [3]]) == [1, 2, 3] and spiral([[1, 2]]) == [1, 2]")

### Exercise: Coding 2: The longest palindrome

Write `longest_palindrome(s)`: the longest substring that reads the same backwards (the leftmost if several), by expanding around each centre: O(n²) time, O(1) extra space.

In [ ]:
def longest_palindrome(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\n\ndef test_palindromes():\n    \"around each centre\"\n    assert longest_palindrome(\"babad\") == \"bab\" and longest_palindrome(\"cbbd\") == \"bb\"\n    assert longest_palindrome(\"\") == \"\" and longest_palindrome(\"a\") == \"a\" and longest_palindrome(\"racecars\") == \"racecar\"\n\ndef test_fast():\n    \"2,000 characters\"\n    s = \"ab\" * 1000\n    start = time.perf_counter()\n    assert len(longest_palindrome(s)) == 1999 and time.perf_counter() - start < 3")

### Exercise: Design: what a CDN saves

Waymark serves map tiles. Write `cdn_savings(views_per_day, tiles_per_view, kb_per_tile, hit_rate, origin_per_server_mbps)`: return a dict with `peak_tiles_per_s` (tiles a day / 86,400 × 4, rounded), `origin_gb_per_day` without a CDN and `origin_gb_per_day_with_cdn` (only misses reach the origin; GB = 10^6 kB; 1 decimal place), and `origin_servers` with and without the CDN at peak, as a tuple `(without, with)`: peak megabits a second (tiles × kB × 8 / 1000) over the server's capacity, rounded up, at least 1.

In [ ]:
import math

def cdn_savings(views_per_day, tiles_per_view, kb_per_tile, hit_rate, origin_per_server_mbps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cdn():\n    \"most requests never reach the origin\"\n    result = cdn_savings(2_000_000, 30, 20, 0.95, 1000)\n    assert result == {\"peak_tiles_per_s\": 2778, \"origin_gb_per_day\": 1200.0, \"origin_gb_per_day_with_cdn\": 60.0, \"origin_servers\": (1, 1)}\n    big = cdn_savings(200_000_000, 30, 20, 0.95, 1000)\n    assert big[\"origin_servers\"] == (45, 3)")

### Bug hunt: A dictionary that changed while looping

Removing keys while looping over the dictionary raises `RuntimeError: dictionary changed size during iteration`. Loop over a copy of the keys so it prints `{'Mill': 40, 'Ridge': 310}`.

In [ ]:
heights = {"Mill": 40, "Island": None, "Ridge": 310, "Moor": None}
for name in heights:
    if heights[name] is None:
        del heights[name]
print(heights)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"iterate over a snapshot\"\n    expect_output(\"{'Mill': 40, 'Ridge': 310}\")")

### Bug hunt: A backup that changed too

The "backup" of the grid is a shallow copy, so editing the grid edits it as well and this prints `9`. Make a deep copy so it prints `0`.

In [ ]:
import copy

grid = [[0, 0], [0, 0]]
backup = grid.copy()
grid[0][0] = 9
print(backup[0][0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"copy the inner lists too\"\n    expect_output(\"0\")")

### Exercise: Debugging: pages that overlapped

`page(items, number, size)` should return page `number` (counting from 1) of `items`, `size` per page, but its slice is off by one and pages overlap. Fix it; a page past the end is empty, and a number below 1 raises `ValueError`.

In [ ]:
def page(items, number, size):
    start = (number - 1) * size
    return items[start:start + size + 1]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pages():\n    \"no overlap, no gaps\"\n    items = list(range(1, 11))\n    assert page(items, 1, 3) == [1, 2, 3] and page(items, 2, 3) == [4, 5, 6] and page(items, 4, 3) == [10] and page(items, 5, 3) == []\n    assert sum((page(items, n, 4) for n in range(1, 4)), []) == items\n    try:\n        page(items, 0, 3)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"page 0\")")

### Exercise: Deep dive: plugins that register themselves

Write a base class `Exporter` so that every subclass declaring a class attribute `format` (such as `format = "gpx"`) is registered automatically in `Exporter.registry` by that format, using `__init_subclass__`. Subclasses without a `format` aren't registered; a second subclass with an already registered format raises `TypeError("duplicate format: F")`. Also write `Exporter.for_format(name)` returning a new instance of the registered class, or raising `KeyError`.

In [ ]:
class Exporter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_registry():\n    \"no list to keep up to date\"\n    class GPX(Exporter):\n        format = \"gpx\"\n    class GeoJSON(Exporter):\n        format = \"geojson\"\n    class Helper(Exporter):\n        pass\n    assert Exporter.registry[\"gpx\"] is GPX and \"Helper\" not in str(Exporter.registry)\n    assert isinstance(Exporter.for_format(\"geojson\"), GeoJSON)\n    try:\n        class AlsoGPX(Exporter):\n            format = \"gpx\"\n    except TypeError as e:\n        assert str(e) == \"duplicate format: gpx\"\n    else:\n        raise AssertionError(\"duplicate\")\n    try:\n        Exporter.for_format(\"kml\")\n    except KeyError:\n        pass\n    else:\n        raise AssertionError(\"kml\")")

> **Behavioural: a time you changed your mind.** Two minutes, aloud, with STAR: "Tell me about a time you changed your mind because of evidence." (Your capstone's user test, or a review comment that turned out to be right, are good candidates.)

### Puzzle: Two centres

Arrange the centre expansion in `longest_palindrome`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # spiral([[1, 2], [3, 4]])
q = ___   # longest_palindrome("abacdc")
r = ___   # True or False: dict.copy() copies the values inside it too
s = ___   # the method Python calls on a class whenever it's subclassed
t = ___   # page(list(range(10)), 2, 4)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2, 4, 3] and q == \"aba\" and r is False, \"p, q and r\"\n    assert s == \"__init_subclass__\" and t == [4, 5, 6, 7], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

A shallow copy is a new outer list holding the **same** inner lists. One of today's debugging bugs is exactly this, in a program that thought it had saved the map before editing it.

## How to run the loop

Start the timer, then take the rounds in order, about 25 minutes each, out loud if you can. For each problem: clarify, examples, simple, better, code, test, complexity. Write a line of notes per round as you finish (what went well, what didn't): you'll need them on Saturday.

## The loop

### Coding round

### Design round

### Debugging round

### Python deep-dive round

### Behavioural round

## Quiz

## Scoring yourself

Score each round from 1 to 4 on its criteria (coding: problem solving, code quality, testing, communication; design: requirements, estimates, trade-offs; debugging: method, fix; deep dive: accuracy, examples; behavioural: structure, ownership, result). Keep the scores: Saturday's debrief compares this loop with tomorrow's.

## Before you go

Tomorrow is another full loop with different problems. Rest well tonight: stamina is part of it.